# SINACON — trích dữ liệu một lần

**Để làm gì:** log thô rất nặng (mỗi tủ vài trăm MB). Notebook này đọc log **một lần**, giữ lại đúng phần các bước sau cần, cất ra file nhỏ (khoảng 5 MB mỗi tủ). Sau đó tính cửa sổ thời gian, chạy mô hình… chỉ đọc file nhỏ này, mất vài giây.

**Giữ lại gì (mỗi tủ một file):**
- mỗi lần trạng thái / mã lỗi của APU và cả tủ thay đổi (APU Stat Trig, APS Stat Trig);
- warning của tủ;
- nhiệt độ mỗi phút của 4 APU (APU Stat 60s).

**Đọc từ:** `Trung Nam/01_log/<đợt>` (chỉ đọc). **Ghi vào:** `MyDrive/SINACON_mo_hinh_lai/bang_trung_gian/<đợt>/`.

**Cách chạy:** ô 1 → ô 2 → ô 3. Ô 2 lâu (khoảng 40 giây mỗi tủ, cả đợt khoảng 1 giờ). Drive ngắt giữa chừng thì chạy lại ô 1 và ô 2: tủ nào xong rồi sẽ bỏ qua, làm tiếp chỗ dở.

In [ ]:
#@title 1. Kết nối Drive và lấy code
import subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
# Code nằm ở nhánh fault-diagnosis (repo công khai, không cần token)
REPO_URL = 'https://github.com/phanthutrang410/OPTIMA-Core.git'
BRANCH = 'fault-diagnosis'
CODE_DIR = Path('/content/OPTIMA-Core')
cmds = ([['git', '-C', str(CODE_DIR), 'fetch', 'origin', BRANCH],
         ['git', '-C', str(CODE_DIR), 'checkout', '-B', BRANCH, 'origin/' + BRANCH]]
        if (CODE_DIR / '.git').is_dir() else [['git', 'clone', '--branch', BRANCH, REPO_URL, str(CODE_DIR)]])
for c in cmds:
    subprocess.run(c, check=True)
sys.path.insert(0, str(CODE_DIR / 'src'))
import importlib, sinacon.extract
importlib.reload(sinacon.extract)
print('Code version:', subprocess.check_output(['git', '-C', str(CODE_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
#@title 2. Trích dữ liệu một đợt (lâu: khoảng 40 giây mỗi tủ)
DOT = 'Tháng 4–6/2026 (04/04–03/06)'  #@param ['Tháng 4–6/2026 (04/04–03/06)', 'Tháng 1–3/2026 (12/01–12/03)', 'Tháng 6–8/2026 (13/06–13/08)', 'Tháng 8–10/2026 (13/08–31/10)', 'Tháng 10/2025']
CHI_TU = ''  #@param {type:'string'}
# Để trống = mọi tủ. Muốn thử vài tủ trước: gõ vd 1.1, 4.1
from sinacon import explore as ex, extract as tr
ROOT = ex.find_period('/content/drive', '', DOT)
OUT = Path('/content/drive/MyDrive/SINACON_mo_hinh_lai/bang_trung_gian') / ex.PERIODS[DOT][2]
print('Đọc từ:', ROOT)
print('Ghi vào:', OUT)
tu = [x.strip() for x in CHI_TU.split(',') if x.strip()] or None
bang = tr.extract_period(ROOT, OUT, tu)


In [ ]:
#@title 3. Đã trích được gì
bang = tr.summary(OUT)
print(f'{len(bang)} tủ đã trích trong {OUT}')
sai = bang[~bang['dung_ma_aps']]
if len(sai):
    print('!!! Thư mục xếp nhầm tủ (mã APS trong file khác với tủ):'); display(sai[['tu', 'aps']])
display(bang)
